# 📊 Tech-Enabled America Fund — Portfolio Analytics
## Notebook 4 · Efficient Frontier & Portfolio Optimisation

Reproduces the mean-variance optimisation from your assignment (`ASS1` Excel `Optimatization` sheet).

**Ex-ante (at construction, Jan 2016 inputs):**
- Expected Return: **17.08%** · Std Dev: **13.67%** · Sharpe: **1.10**

**Realised (2017–2022):**
- CAGR: **22.50%** · Std Dev: **6.33%** · Sharpe: **3.22**

The fund significantly outperformed its ex-ante forecast — driven by NVDA and the broader tech supercycle.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.optimize import minimize
import json, warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (12, 7), 'axes.spines.top': False,
    'axes.spines.right': False, 'axes.grid': True,
    'grid.alpha': 0.25, 'font.size': 11, 'axes.titlesize': 13,
})
BLUE, RED, GREEN, GOLD = '#1a5276', '#c0392b', '#1e8449', '#d4ac0d'

stock_ret = pd.read_csv('../data/stock_monthly_returns.csv', index_col=0, parse_dates=True)
with open('../data/portfolio_config.json') as f:
    PORTFOLIO = json.load(f)
with open('../data/confirmed_metrics.json') as f:
    M = json.load(f)

RF_ANN = M['RF']
TICKERS = [t for t in PORTFOLIO.keys() if t in stock_ret.columns]
W_ACT   = np.array([PORTFOLIO[t]['weight'] for t in TICKERS])

# Annualised inputs from realised 2017-2022 data
rets = stock_ret[TICKERS]
mu   = rets.mean().values  * 12
cov  = rets.cov().values   * 12

print(f'Tickers: {TICKERS}')
print(f'Annualised returns (realised):  {(mu*100).round(1)}')

## 1 · Portfolio Statistics

In [ ]:
def port_stats(w, mu=mu, cov=cov, rf=RF_ANN):
    r = w @ mu
    v = np.sqrt(w @ cov @ w)
    s = (r - rf) / v
    return r, v, s

r_act, v_act, s_act = port_stats(W_ACT)

print('─'*55)
print('  Actual Fund (assignment weights) — Realised inputs')
print('─'*55)
print(f'  Annualised Return:  {r_act*100:.2f}%')
print(f'  Annualised Vol:     {v_act*100:.2f}%')
print(f'  Sharpe Ratio:       {s_act:.4f}')
print()
print('  Confirmed realised performance (2017-2022):')
print(f'  CAGR: {M["CAGR"]*100:.2f}%  |  Monthly Std: {M["Annualised_Vol"]*100:.2f}%  |  Sharpe: {M["Sharpe"]:.4f}')

## 2 · Monte Carlo Simulation

In [ ]:
np.random.seed(42)
N, n = 15_000, len(TICKERS)
sim_r = np.zeros(N); sim_v = np.zeros(N); sim_s = np.zeros(N)

for i in range(N):
    w = np.random.dirichlet(np.ones(n))  # sums to 1, all positive
    r, v, s = port_stats(w)
    sim_r[i] = r; sim_v[i] = v; sim_s[i] = s

print(f'Simulated {N:,} random portfolios')
print(f'Sharpe range: {sim_s.min():.2f} → {sim_s.max():.2f}')
print(f'Return range: {sim_r.min()*100:.1f}% → {sim_r.max()*100:.1f}%')

## 3 · Optimised Portfolios

In [ ]:
# Weight constraints matching assignment (from ASS1 Optimatization sheet)
# MinWe / MaxWe columns from the Excel
BOUNDS_MAP = {
    'NVDA':  (0.08, 0.25), 'GOOGL': (0.08, 0.17), 'AMZN':  (0.08, 0.17),
    'HON':   (0.05, 0.10), 'MA':    (0.05, 0.10), 'UNH':   (0.04, 0.07),
    'BALL':  (0.03, 0.05), 'NEE':   (0.03, 0.06), 'PG':    (0.02, 0.04),
    'EQIX':  (0.06, 0.12), 'SLB':   (0.02, 0.04),
}
bounds = tuple(BOUNDS_MAP.get(t, (0.02, 0.30)) for t in TICKERS)
cons   = [{'type': 'eq', 'fun': lambda w: w.sum() - 1}]
w0     = W_ACT

# Maximum Sharpe Ratio (tangency portfolio)
res_ms = minimize(lambda w: -port_stats(w)[2], w0, method='SLSQP', bounds=bounds, constraints=cons)
r_ms, v_ms, s_ms = port_stats(res_ms.x)

# Minimum Variance
res_mv = minimize(lambda w: port_stats(w)[1], w0, method='SLSQP', bounds=bounds, constraints=cons)
r_mv, v_mv, s_mv = port_stats(res_mv.x)

print('─'*58)
print(f'  Portfolio         Return     Volatility  Sharpe')
print('─'*58)
print(f'  Actual Fund       {r_act*100:5.2f}%     {v_act*100:5.2f}%     {s_act:.3f}')
print(f'  Max Sharpe        {r_ms*100:5.2f}%     {v_ms*100:5.2f}%     {s_ms:.3f}')
print(f'  Min Variance      {r_mv*100:5.2f}%     {v_mv*100:5.2f}%     {s_mv:.3f}')

In [ ]:
# ── Efficient Frontier ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 7))

# Random portfolios coloured by Sharpe
sc = ax.scatter(sim_v*100, sim_r*100, c=sim_s, cmap='viridis', s=4, alpha=0.3, zorder=1)
plt.colorbar(sc, ax=ax, label='Sharpe Ratio', shrink=0.75)

# Individual stocks
for i, t in enumerate(TICKERS):
    sv = np.sqrt(cov[i,i]) * 100
    sr = mu[i] * 100
    ax.scatter(sv, sr, marker='D', s=55, color='#888', zorder=4, alpha=0.85)
    ax.annotate(t, xy=(sv, sr), xytext=(4, 3), textcoords='offset points',
                fontsize=9, color='#444')

# Key portfolios
ax.scatter(v_ms*100, r_ms*100, marker='*', s=420, color=GOLD,  zorder=6,
           edgecolors='black', lw=0.5, label=f'Max Sharpe  (S={s_ms:.2f})')
ax.scatter(v_mv*100, r_mv*100, marker='^', s=230, color=RED,   zorder=6,
           edgecolors='black', lw=0.5, label='Min Variance')
ax.scatter(v_act*100, r_act*100, marker='P', s=280, color=BLUE, zorder=6,
           edgecolors='white',  lw=1.0, label=f'Actual Fund  (S={s_act:.2f})')

# Capital Market Line
cml_x = np.linspace(0, v_ms * 1.8, 100) * 100
cml_y = (RF_ANN + s_ms * cml_x/100) * 100
ax.plot(cml_x, cml_y, color=GOLD, lw=1.5, linestyle='--', alpha=0.85, label='Capital Market Line')

ax.set_xlabel('Annualised Volatility (%)')
ax.set_ylabel('Annualised Return (%)')
ax.set_title('Efficient Frontier — Tech-Enabled America Fund · Realised Data (2017–2022)', fontweight='bold')
ax.legend(fontsize=9, loc='upper left')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
plt.tight_layout()
plt.savefig('../data/efficient_frontier.png', dpi=150, bbox_inches='tight')
plt.show()

## 4 · Weight Comparison

In [ ]:
wt_df = pd.DataFrame({
    'Actual (ASS1 optimised)': W_ACT * 100,
    'Max Sharpe (realised)':   res_ms.x * 100,
    'Min Variance (realised)': res_mv.x * 100,
}, index=TICKERS).round(2)

print('Portfolio Weights (%):')
print(wt_df.to_string())

fig, ax = plt.subplots(figsize=(13, 5))
x, w = np.arange(len(TICKERS)), 0.27
colors = [BLUE, GOLD, RED]
for i, (col, color) in enumerate(zip(wt_df.columns, colors)):
    ax.bar(x + i*w, wt_df[col], width=w, label=col,
           color=color, alpha=0.85, edgecolor='white')

ax.set_xticks(x + w); ax.set_xticklabels(TICKERS)
ax.set_ylabel('Weight (%)')
ax.set_title('Portfolio Weights: Assignment vs Realised-Data Optimisation', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../data/weight_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nNotebook 04 complete ✓')
print('All 4 notebooks done. Charts saved to data/ folder.')